In [10]:
import numpy as np
import bvhio
import warnings
import json
import re
import xml.etree.ElementTree as ET
from ultralytics import YOLO
from scipy.spatial.transform import Rotation as R
from pymvg.camera_model import CameraModel
from pymvg.multi_camera_system import MultiCameraSystem

warnings.filterwarnings('ignore')

sequences_file_path = 'gait3d\\ListOfSequences.txt'
yolo_model_path = "yolo26x-pose.pt"

In [11]:
def parse_sequences(file_path: str) -> dict:
    sequence_dict = {}

    details_pattern = re.compile(
        r"start frame: (\d+), number of frames: (\d+), frames offset: (-?\d+), MoCap data: (\w+)"
    )

    with open(file_path, 'r') as file:
        for line in file:
            if line.strip().startswith('* p'):
                last_key = line.split(' ')[1]
            
            detail_match = details_pattern.search(line.strip())
            if detail_match:
                sequence_dict[last_key] = {
                    "start_frame": int(detail_match.group(1)),
                    "number_of_frames": int(detail_match.group(2)),
                    "frame_offset": int(detail_match.group(3)),
                    "MoCap_data": detail_match.group(4) == "Yes"
                }
                
    return sequence_dict

In [12]:
def get_video_files(sequence_key):
    file_path = './gait3d/ListOfSequences.txt'
    sequence_info = parse_sequences(file_path)[sequence_key]
    avi_file_names =[
        f"c{camera_number}_{(4 - len(str(sequence_info['start_frame']))) * '0' + str(sequence_info['start_frame'])}" 
        for camera_number in range(1, 5)
        ]
    
    avi_seq_paths = [
        f"./gait3d/Sequences/{sequence_key}/Images/{avi_file_name}.avi"
        for avi_file_name in avi_file_names
        ]
    
    return avi_seq_paths


def get_camera_calibration_files(sequence_key):
    calibraton_file_path_base = "./gait3d/Sequences/{sequence_key}/Calibration/c{camera_number}.xml"
    camera_file_paths = [calibraton_file_path_base.format(sequence_key=sequence_key, camera_number=c_num) for c_num in range(1, 5)]
    return camera_file_paths


In [13]:
def parse_camera_xml(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()

    camera_name = root.attrib['name']

    geometry = root.find('Geometry')
    intrinsic = root.find('Intrinsic')
    extrinsic = root.find('Extrinsic')
    
    camera_width = float(geometry.get('width'))
    camera_height = float(geometry.get('height'))

    f = float(intrinsic.get('focal'))
    c = np.array([
        float(intrinsic.get('cx')),
        float(intrinsic.get('cy'))
    ])

    k = np.array([
        float(intrinsic.get('kappa1')), 0.0, 0.0
        # k2=0 and k3=0 - not provided in camera config file
    ])

    p = np.array([0.0, 0.0])  # p - not provided in camera config file

    T = np.array([
        [float(extrinsic.get("tx"))],
        [float(extrinsic.get("ty"))],
        [float(extrinsic.get("tz"))]
    ]).reshape(3, 1)

    rx, ry, rz = float(extrinsic.get("rx")), float(extrinsic.get("ry")), float(extrinsic.get("rz"))
    R_matrix = R.from_euler('xyz', [rx, ry, rz]).as_matrix()

    return {
        "name": camera_name,
        "width": camera_width,
        "height": camera_height,
        "R": R_matrix,
        "T": T,
        "f": f,
        "c": c,
        "k": k,
        "p": p
    }


In [14]:
def unfold_camera_param(camera):
    """
    Camera parameters:
        R: 3x3 Camera rotation matrix
        T: 3x1 Camera translation parameters
        f: (scalar) Camera focal length
        c: 2x1 Camera center
        k: 3x1 Camera radial distortion coefficients
        p: 2x1 Camera tangential distortion coefficients
    """
    R = camera['R']
    T = camera['T']
    f = camera['f'] 
    c = camera['c']
    k = camera['k']
    p = camera['p']
    return R, T, f, c, k, p


def build_multi_camera_system(cameras):
    """
    Build a multi-camera system with pymvg package for triangulation

    Args:
        cameras: list of camera parameters
    Returns:
        cams_system: a multi-cameras system
    """
    pymvg_cameras = []
    for camera in cameras:
        R, T, f, c, k, p = unfold_camera_param(camera)
        camera_matrix = np.array(
            [[f, 0, c[0]], [0, f, c[1]], [0, 0, 1]], dtype=float)
        distortion = np.array([k[0], k[1], p[0], p[1], k[2]])
        distortion.shape = (5,)
        M = camera_matrix.dot(np.concatenate((R, T), axis=1))
        camera = CameraModel.load_camera_from_M(
            M, name=camera['name'], distortion_coefficients=distortion,
            width=camera['width'], height=camera['height']
        )
        pymvg_cameras.append(camera)
    return MultiCameraSystem(pymvg_cameras)


def triangulate_one_point(camera_system, points_2d_set):
    """
    Triangulate 3d point in world coordinates with multi-views 2d points

    Args:
        camera_system: pymvg camera system
        points_2d_set: list of structure (camera_name, point2d)
    Returns:
        points_3d: 3x1 point in world coordinates
    """
    points_3d = camera_system.find3d(points_2d_set)
    return points_3d


def triangulate_poses(cameras_params, poses2d):
    """
    Triangulate 3d points in world coordinates of multi-view 2d poses
    by interatively calling $triangulate_one_point$

    Args:
        camera_params: a list of camera parameters, each corresponding to single camera
        poses2d: ndarray of shape nxkx2, len(cameras) == n
    Returns:
        poses3d: ndarray of shape n/nviews x k x 3
    """
    nviews = poses2d.shape[0]
    njoints = poses2d.shape[1]
    ninstances = 1 
    
    poses3d = []
    for i in range(ninstances):
        camera_system = build_multi_camera_system(cameras_params)

        pose3d = np.zeros((njoints, 3))
        for k in range(njoints):
            points_2d_set = []

            for j in range(nviews):
                camera_name = cameras_params[j]['name']
                points_2d = poses2d[i * nviews + j, k, :]
                points_2d_set.append((camera_name, points_2d))
            pose3d[k, :] = triangulate_one_point(camera_system, points_2d_set).T
        poses3d.append(pose3d)
    return np.array(poses3d)

# based on https://github.com/microsoft/multiview-human-pose-estimation-pytorch/blob/master/lib/multiviews/triangulate.py

In [15]:
model = YOLO(yolo_model_path)
sequences = parse_sequences(sequences_file_path)

In [16]:
SEQ_KEY = 'p1s1'
FRAME_WIDTH = 960
FRAME_HEIGHT = 540
YOLO_LANDMARKS_NUM = 17

if sequences[SEQ_KEY]['MoCap_data']:
    video_files = get_video_files(SEQ_KEY)
    max_frames = sequences[SEQ_KEY]['number_of_frames']
            
    camera_files_paths = get_camera_calibration_files(SEQ_KEY)
    cameras_params = [parse_camera_xml(camera_path) for camera_path in camera_files_paths]

    yolo_predictions = {f"c{i+1}": {} for i in range(4)}
       
    for c_idx, c_file in enumerate(video_files):
        results = model.predict(
            source=c_file,
            show=False,
            save=False,
            project='gait_3d',
            name='yolo26', 
            verbose=False, 
            stream=True
        )

        
        for f_idx, result in enumerate(results):
            xy_n = result.keypoints.xyn[0].cpu().numpy().tolist()
            yolo_predictions[f"c{c_idx+1}"][f_idx] = xy_n
            
    triangulation_results = []
    
    for f_idx in range(max_frames):
        found_2d_points = np.array([np.array(yolo_predictions[f"c{camera_i+1}"][f_idx]) * [FRAME_WIDTH, FRAME_HEIGHT] for camera_i in range(4)])
        frame_triangulation_result = triangulate_poses(cameras_params, found_2d_points)
        triangulation_results.append(frame_triangulation_result[0].tolist())


In [17]:
triangulation_results[0]

[[-129.7677272848799, 2305.497479427381, 1452.9159489306542],
 [-105.91440443260916, 2314.985061291976, 1484.6095675038944],
 [-153.60405755581246, 2313.6008628181044, 1485.279126430626],
 [-48.727764582569726, 2373.3645743793054, 1476.6452386537005],
 [-203.86993691837264, 2374.4600523698755, 1480.2047923462926],
 [23.562321914630456, 2427.624415646028, 1285.4478886879278],
 [-269.60076682828156, 2428.604691874187, 1289.8733322387773],
 [51.05545527337205, 2447.333352599239, 1016.4376381697089],
 [-299.57684774422154, 2447.4924335670175, 1025.162836746703],
 [61.31619881223193, 2445.9370375700923, 796.476232519037],
 [-316.7582120719547, 2439.183496684828, 803.8052616909613],
 [-31.32170853639191, 2424.3527395983556, 844.653977994504],
 [-226.60728799876665, 2421.523809290233, 846.5302619891],
 [-40.83100736868302, 2498.9412133997475, 439.61683852064056],
 [-222.51090956748274, 2490.033783137874, 438.73268542637186],
 [-47.57161901203288, 2552.503525796675, 54.79564449034536],
 [-209.

In [18]:
yolo_predictions['c1'][0]

[[0.9111762046813965, 0.2457217127084732],
 [0.9177706241607666, 0.23574523627758026],
 [0.9099023938179016, 0.23544980585575104],
 [0.9361823201179504, 0.23713763058185577],
 [0.9124893546104431, 0.2360096424818039],
 [0.9481129050254822, 0.2982827425003052],
 [0.9106159806251526, 0.2922152876853943],
 [0.9483572244644165, 0.38139453530311584],
 [0.9033466577529907, 0.36769118905067444],
 [0.9411687850952148, 0.449741005897522],
 [0.8957064747810364, 0.4262283146381378],
 [0.9276393055915833, 0.4293688237667084],
 [0.9021046161651611, 0.42261838912963867],
 [0.9265609979629517, 0.5447744727134705],
 [0.9039694666862488, 0.5361360907554626],
 [0.92339688539505, 0.6515582203865051],
 [0.9003700613975525, 0.6378198862075806]]